In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import tiktoken
from datasets import load_dataset

import time
from components.dataset import TextDataset
from components.model import GPTModel
from components.tokenizer import tokenizer as final_tokenizer
from tqdm import tqdm

/home/software/Documents/projects/thonk-v1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_encoding= tiktoken.get_encoding("r50k_base")
special_tokens={
  "[INST]": base_encoding.n_vocab,
  "[/INST]": base_encoding.n_vocab + 1,
  "[STARTOFTEXT]": base_encoding.n_vocab + 2,
  "[ENDOFTEXT]": base_encoding.n_vocab + 3,
}
tokenizer = tiktoken.Encoding(
  name="bob",
  pat_str=base_encoding._pat_str,
  mergeable_ranks=base_encoding._mergeable_ranks,
  special_tokens={**base_encoding._special_tokens, **special_tokens}
)
def encode(text):
  return tokenizer.encode(text, allowed_special={"[INST]", "[/INST]", "[STARTOFTEXT]", "[ENDOFTEXT]"})

def decode(text):
  return tokenizer.decode(text)
text="[STARTOFTEXT] hello world [ENDOFTEXT]"
print(encode(text))
print(decode(encode(text)))

[50259, 23748, 995, 220, 50260]
[STARTOFTEXT] hello world [ENDOFTEXT]


In [3]:
base_dataset = load_dataset("HuggingFaceFW/fineweb-edu", "default", streaming=True)
# next_text = next(iter(base_dataset['train']['text']))

In [4]:
block_size = 256
n_embedding = 256
n_layers = 8
n_heads = 8
dropout_p = 0.1
batch_size=32
learning_rate = 1e-4
max_iters = 1000
pbar_update_interval = 2
device="cuda" if torch.cuda.is_available() else "cpu"

In [5]:
dataset = TextDataset(base_dataset, block_size)

# for i in range(10):
#   start = time.perf_counter()
#   x = (next(iter(dataset['train'])))
#   end = time.perf_counter()
  # print(f"time {i}: {end-start}")

In [6]:
model = GPTModel(block_size=block_size, n_embedding=n_embedding, n_layers=n_layers, n_heads=n_heads, dropout_p=dropout_p, vocab_size=final_tokenizer.n_vocab)
text_ctx = torch.tensor(encode("the output vector is B,T,logits")).unsqueeze(0)
model(text_ctx).shape

torch.Size([1, 10, 50261])

In [7]:
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
dataloader = DataLoader(dataset, batch_size=batch_size)
vocab_size = final_tokenizer.n_vocab


pbar = tqdm(total=max_iters)
data_iter = iter(dataloader)
for i in range(max_iters):
  try:
    xb, yb = next(data_iter)
  except StopIteration:
    data_iter = iter(dataloader)
    xb, yb = next(data_iter)
  xb, yb = xb.to(device), yb.to(device)

  logits = model(xb)
  # logits = logits.argmax(dim=-1)
  logits = logits.transpose(1,2)
  loss = loss_fn(logits, yb)
  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  optimizer.step()
  if i%pbar_update_interval ==0:
    pbar.set_postfix({'loss':f'{loss.item():.4f}'})
    pbar.update(pbar_update_interval)

  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_3948198/2277925481.py:24: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(model.parameters(), max_norm=1.0)
100%|██████████| 1000/1000 [03:02<00:00,  5.78it/s, loss=7.0411]

In [8]:
def generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8, top_k=None):
    """Generate text from the model given a prompt."""
    model.eval()
    with torch.no_grad():
        # Encode prompt
        tokens = torch.tensor(encode(prompt)).unsqueeze(0).to(device)
        
        # Generation loop
        for _ in range(max_new_tokens):
            # Crop context to block_size
            tokens_cond = tokens if tokens.size(1) <= block_size else tokens[:, -block_size:]
            
            # Forward pass
            logits = model(tokens_cond)
            
            # Focus on last position
            logits = logits[:, -1, :] / temperature
            
            # Optional: top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            # Get probabilities and sample
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            tokens = torch.cat([tokens, next_token], dim=1)
    
    model.train()
    return decode(tokens[0].tolist())

# After the training loop:
generated = generate_text(model, tokenizer, "The meaning of life is", max_new_tokens=100)
print("\n--- Generated Text ---")
print(generated)



--- Generated Text ---
The meaning of life is reinse of a December and form,f, and other local to the number of the abiding.
lis of the pre- If they that is Dynamic againstfieldling of the carried of we have been also only the week a increasing and in the area uniform of the number of the trunk.
-3 and the test, and the Warren of the most Constitution of the point, and Rights, and wisdom of the more than iteters and the future of the domain.
Theamation of Bi
